# Rock Paper Scissors Extended Project - Solution Notebook

**Based on Al Sweigart's *The Big Book of Small Python Projects* (Project #59)**

This solution implements the classic game, adds Rock-Paper-Scissors-Lizard-Spock (RPSLS),
uses both if-elif and dictionary-based win logic (alternate), supports a points / double-or-nothing mode,
includes more practice exercises, and a Monte-Carlo simulation section with plots.

---
## Flowchart of the Desired Outcome

In [1]:
from IPython.display import Image, display
display(Image(filename='rock_paper_scissors_flowchart.png', width=900))

## 1. Imports and Core Constants

Import required modules and define the move mappings for classic RPS and full RPSLS.

In [2]:
import random
import time
import sys
from collections import Counter
import matplotlib.pyplot as plt

# Classic RPS mapping: letter -> full name
CLASSIC_MOVES = {
    'R': 'ROCK',
    'P': 'PAPER',
    'S': 'SCISSORS'
}

# RPSLS (Lizard + Spock).  V = Spock (to avoid conflict with S)
RPSLS_MOVES = {
    'R': 'ROCK',
    'P': 'PAPER',
    'S': 'SCISSORS',
    'L': 'LIZARD',
    'V': 'SPOCK'
}

print('Classic moves:', list(CLASSIC_MOVES.keys()))
print('RPSLS moves:', list(RPSLS_MOVES.keys()))
print('Rules (classic):')
print('  Rock beats scissors.')
print('  Paper beats rock.')
print('  Scissors beats paper.')

Classic moves: ['R', 'P', 'S']
RPSLS moves: ['R', 'P', 'S', 'L', 'V']
Rules (classic):
  Rock beats scissors.
  Paper beats rock.
  Scissors beats paper.


## 2. Win-Condition Tables (Two Implementations)

### Primary: dictionary of what each move beats (clean & extensible)
### Alternate: pure if-elif chain (matches original book style)

In [3]:
# Primary (dict) – what each move beats
BEATS = {
    'ROCK':     ['SCISSORS', 'LIZARD'],
    'PAPER':    ['ROCK', 'SPOCK'],
    'SCISSORS': ['PAPER', 'LIZARD'],
    'LIZARD':   ['SPOCK', 'PAPER'],
    'SPOCK':    ['SCISSORS', 'ROCK']
}

def wins_dict(player, computer):
    """Return True if player beats computer using the BEATS dict."""
    return computer in BEATS.get(player, [])

# Alternate: classic if-elif (book style)
def wins_ifelif(player, computer):
    """Return True if player beats computer using if-elif chain."""
    if player == computer:
        return False
    if player == 'ROCK' and computer == 'SCISSORS': return True
    if player == 'PAPER' and computer == 'ROCK': return True
    if player == 'SCISSORS' and computer == 'PAPER': return True
    if player == 'ROCK' and computer == 'LIZARD': return True
    if player == 'PAPER' and computer == 'SPOCK': return True
    if player == 'SCISSORS' and computer == 'LIZARD': return True
    if player == 'LIZARD' and computer == 'SPOCK': return True
    if player == 'LIZARD' and computer == 'PAPER': return True
    if player == 'SPOCK' and computer == 'SCISSORS': return True
    if player == 'SPOCK' and computer == 'ROCK': return True
    return False

print('Dict wins ROCK vs SCISSORS:', wins_dict('ROCK', 'SCISSORS'))
print('Dict wins PAPER vs ROCK:', wins_dict('PAPER', 'ROCK'))
print('Dict wins SCISSORS vs PAPER:', wins_dict('SCISSORS', 'PAPER'))
print('Dict wins ROCK vs PAPER:', wins_dict('ROCK', 'PAPER'))
print('If-elif ROCK vs SCISSORS:', wins_ifelif('ROCK', 'SCISSORS'))
print('If-elif LIZARD vs SPOCK:', wins_ifelif('LIZARD', 'SPOCK'))

Dict wins ROCK vs SCISSORS: True
Dict wins PAPER vs ROCK: True
Dict wins SCISSORS vs PAPER: True
Dict wins ROCK vs PAPER: False
If-elif ROCK vs SCISSORS: True
If-elif LIZARD vs SPOCK: True


## 3. Core Helper Functions

In [4]:
def get_computer_move(mode='classic'):
    """Return a random full-name move for the given mode."""
    moves = CLASSIC_MOVES if mode == 'classic' else RPSLS_MOVES
    return random.choice(list(moves.values()))

def normalize_player_input(letter, mode='classic'):
    """Convert single letter to full move name, or None if invalid."""
    moves = CLASSIC_MOVES if mode == 'classic' else RPSLS_MOVES
    return moves.get(letter.upper())

def determine_outcome(player, computer, use_dict=True):
    """Return ('win'|'lose'|'tie', message)"""
    if player == computer:
        return 'tie', "It's a tie!"
    winner_fn = wins_dict if use_dict else wins_ifelif
    if winner_fn(player, computer):
        return 'win', 'You win!'
    else:
        return 'lose', 'You lose!'

random.seed(42)
print('Sample computer moves (classic):', [get_computer_move('classic') for _ in range(5)])
print('Sample computer moves (RPSLS):', [get_computer_move('rpsls') for _ in range(5)])

Sample computer moves (classic): ['PAPER', 'ROCK', 'SCISSORS', 'PAPER', 'ROCK']
Sample computer moves (RPSLS): ['LIZARD', 'SPOCK', 'ROCK', 'PAPER', 'SCISSORS']


## 4. Single Round (Non-Interactive for Jupyter / Simulation)

`play_round` prints the dramatic countdown (optional) and returns the outcome string.

In [5]:
def play_round(player_letter, mode='classic', pause=False, use_dict=True, verbose=True):
    """Play one round. player_letter is 'R','P','S' (or 'L','V'). Returns outcome."""
    player = normalize_player_input(player_letter, mode)
    if player is None:
        if verbose: print('Invalid move.')
        return None
    if verbose: print(f'{player} versus...')
    if pause:
        time.sleep(0.5)
        print('1...')
        time.sleep(0.25)
        print('2...')
        time.sleep(0.25)
        print('3...')
        time.sleep(0.25)
    else:
        print('1...')
        print('2...')
        print('3...')
    computer = get_computer_move(mode)
    if verbose: print(computer)
    outcome, msg = determine_outcome(player, computer, use_dict=use_dict)
    if verbose: print(msg)
    return outcome

random.seed(1)
print(f"Outcome: {play_round('R', pause=False)}")
print()
random.seed(2)
print(f"Outcome: {play_round('P', pause=False)}")

ROCK versus...
1...
2...
3...
SCISSORS
You win!
Outcome: win

PAPER versus...
1...
2...
3...
ROCK
You win!
Outcome: win


## 5. Full Interactive Game Loop (matches original book)

Defined for you to call manually. Uses input() so not auto-executed.

In [6]:
def play_interactive_game(mode='classic', use_dict=True):
    """Exact recreation of the original book program + optional RPSLS."""
    print('''Rock, Paper, Scissors, by Al Sweigart al@inventwithpython.com
- Rock beats scissors.
- Paper beats rock.
- Scissors beats paper.''')
    if mode == 'rpsls':
        print('''- Lizard poisons Spock / eats paper.
- Spock smashes scissors / vaporizes rock.''')
    print()
    wins = losses = ties = 0
    moves = CLASSIC_MOVES if mode == 'classic' else RPSLS_MOVES
    prompt = '(R)ock (P)aper (S)cissors'
    if mode == 'rpsls': prompt += ' (L)izard (V)Spock'
    prompt += ' or (Q)uit'
    while True:
        while True:
            print(f'{wins} Wins, {losses} Losses, {ties} Ties')
            print(f'Enter your move: {prompt}')
            playerMove = input('> ').upper().strip()
            if playerMove == 'Q':
                print('Thanks for playing!')
                return
            if playerMove in moves: break
            print('Type one of ' + ', '.join(moves.keys()) + ', or Q.')
        full = moves[playerMove]
        print(f'{full} versus...')
        time.sleep(0.5)
        print('1...')
        time.sleep(0.25)
        print('2...')
        time.sleep(0.25)
        print('3...')
        time.sleep(0.25)
        computer = get_computer_move(mode)
        print(computer)
        time.sleep(0.5)
        outcome, msg = determine_outcome(full, computer, use_dict=use_dict)
        print(msg)
        if outcome == 'win': wins += 1
        elif outcome == 'lose': losses += 1
        else: ties += 1

print('Interactive loop defined as play_interactive_game(). Call it yourself if you want real input().')

Interactive loop defined as play_interactive_game(). Call it yourself if you want real input().


## 6. Alternate Code – Pure if-elif Win Logic (Book Style)

Self-contained alternate that never uses the BEATS dictionary.

In [7]:
def determine_outcome_ifelif(player, computer):
    """Alternate pure if-elif implementation (matches book spirit)."""
    if player == computer: return 'tie', "It's a tie!"
    if (player == 'ROCK' and computer in ('SCISSORS', 'LIZARD') or
        player == 'PAPER' and computer in ('ROCK', 'SPOCK') or
        player == 'SCISSORS' and computer in ('PAPER', 'LIZARD') or
        player == 'LIZARD' and computer in ('SPOCK', 'PAPER') or
        player == 'SPOCK' and computer in ('SCISSORS', 'ROCK')):
        return 'win', 'You win!'
    return 'lose', 'You lose!'

test_pairs = [('ROCK', 'SCISSORS'), ('PAPER', 'ROCK'), ('SCISSORS', 'PAPER'),
              ('ROCK', 'PAPER'), ('LIZARD', 'SPOCK'), ('SPOCK', 'ROCK'), ('PAPER', 'PAPER')]
for p, c in test_pairs:
    o1, _ = determine_outcome(p, c, use_dict=True)
    o2, _ = determine_outcome_ifelif(p, c)
    assert o1 == o2
    print(f'{p} vs {c} → {o1}')

ROCK vs SCISSORS → win
PAPER vs ROCK → win
SCISSORS vs PAPER → win
ROCK vs PAPER → lose
LIZARD vs SPOCK → win
SPOCK vs ROCK → win
PAPER vs PAPER → tie


## 7. Points System + Double-or-Nothing (Book Suggestion)

Each win awards the current stake (starts at 1). After a win the stake doubles for the next risk.

In [8]:
def play_points_demo(n_rounds=5, seed=99):
    """Non-interactive demonstration of the points / double-or-nothing idea."""
    random.seed(seed)
    points = 0
    stake = 1
    print('=== Points demo (fixed seed) ===')
    for i in range(1, n_rounds + 1):
        player = random.choice(list(CLASSIC_MOVES.values()))
        computer = get_computer_move('classic')
        outcome, _ = determine_outcome(player, computer)
        if outcome == 'win':
            points += stake
            print(f'Round {i}: {player} vs {computer} → win  | points={points} stake={stake*2} (doubled)')
            stake *= 2
        elif outcome == 'lose':
            points -= stake
            print(f'Round {i}: {player} vs {computer} → lose | points={points} stake=1 (reset)')
            stake = 1
        else:
            print(f'Round {i}: {player} vs {computer} → tie  | points={points} stake={stake}')
    print(f'Final points: {points}')
    return points

play_points_demo()

=== Points demo (fixed seed) ===
Round 1: SCISSORS vs PAPER → win  | points=1 stake=2 (doubled)
Round 2: ROCK vs ROCK → tie  | points=1 stake=2
Round 3: PAPER vs SCISSORS → lose | points=-1 stake=1 (reset)
Round 4: SCISSORS vs ROCK → lose | points=-2 stake=1 (reset)
Round 5: ROCK vs PAPER → lose | points=-3 stake=1 (reset)
Final points: -3


## 8. More Practice Exercises (with solutions)

### Practice A – Best-of-N series
Play until one side reaches n wins (ties ignored).

In [9]:
def best_of_n(n=3, mode='classic', seed=None):
    """Play until one side has n wins. Returns (player_wins, computer_wins)."""
    if seed is not None: random.seed(seed)
    p_wins = c_wins = 0
    letters = list((CLASSIC_MOVES if mode == 'classic' else RPSLS_MOVES).keys())
    while p_wins < n and c_wins < n:
        p = normalize_player_input(random.choice(letters), mode)
        c = get_computer_move(mode)
        outcome, _ = determine_outcome(p, c)
        if outcome == 'win': p_wins += 1
        elif outcome == 'lose': c_wins += 1
    return p_wins, c_wins

pw, cw = best_of_n(3, seed=7)
print(f'Best-of-5 (first to 3) result: player {pw} - computer {cw}')

Best-of-5 (first to 3) result: player 3 - computer 1


### Practice B – Always-Win version (Project #60 idea)
Computer always chooses the losing move.

In [10]:
def always_lose_move(player_full):
    """Return a move that loses to the given player move (classic only)."""
    losers = {'ROCK': 'SCISSORS', 'PAPER': 'ROCK', 'SCISSORS': 'PAPER'}
    return losers[player_full]

def play_always_win_round(player_letter):
    player = normalize_player_input(player_letter, 'classic')
    computer = always_lose_move(player)
    print(f'You play {player} → computer forced to {computer} → You win!')
    return 'win'

print('Always-win demo:')
for letter in 'RPS':
    play_always_win_round(letter)

Always-win demo:
You play ROCK → computer forced to SCISSORS → You win!
You play PAPER → computer forced to ROCK → You win!
You play SCISSORS → computer forced to PAPER → You win!


## 9. Simulation Section – Modify Parameters & Observe Results

Change `N_GAMES`, `PLAYER_STRATEGY`, and `MODE` below, then re-run to see different win-rate distributions and charts.

In [11]:
# ========== TUNABLE PARAMETERS ==========
N_GAMES = 5000          # how many rounds to simulate
PLAYER_STRATEGY = 'random'  # 'random' | 'always_rock' | 'cycle' | 'always_paper'
MODE = 'classic'        # 'classic' | 'rpsls'
SEED = 123
# ========================================

def player_strategy_move(strategy, round_idx, mode):
    moves = list((CLASSIC_MOVES if mode == 'classic' else RPSLS_MOVES).values())
    if strategy == 'random': return random.choice(moves)
    elif strategy == 'always_rock': return 'ROCK'
    elif strategy == 'always_paper': return 'PAPER'
    elif strategy == 'cycle': return moves[round_idx % len(moves)]
    else: return random.choice(moves)

def run_simulation(n_games=N_GAMES, strategy=PLAYER_STRATEGY, mode=MODE, seed=SEED):
    random.seed(seed)
    results = []
    for i in range(n_games):
        p = player_strategy_move(strategy, i, mode)
        c = get_computer_move(mode)
        outcome, _ = determine_outcome(p, c)
        results.append(outcome)
    return Counter(results)

counts = run_simulation()
total = sum(counts.values())
print(f'Simulation: {N_GAMES} games | mode={MODE} | strategy={PLAYER_STRATEGY}')
print(f"Wins:   {counts['win']:5d} ({100*counts['win']/total:.1f}%)")
print(f"Losses: {counts['lose']:5d} ({100*counts['lose']/total:.1f}%)")
print(f"Ties:   {counts['tie']:5d} ({100*counts['tie']/total:.1f}%)")

Simulation: 5000 games | mode=classic | strategy=random
Wins:   1678 (33.6%)
Losses: 1659 (33.2%)
Ties:   1663 (33.3%)


In [12]:
# Visualize the simulation
labels = ['Win', 'Lose', 'Tie']
values = [counts['win'], counts['lose'], counts['tie']]
colors = ['#27AE60', '#E74C3C', '#F39C12']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(labels, values, color=colors, edgecolor='black')
axes[0].set_title(f'Outcome Counts (n={N_GAMES}, strategy={PLAYER_STRATEGY})')
axes[0].set_ylabel('Count')
for i, v in enumerate(values):
    axes[0].text(i, v + max(values)*0.01, str(v), ha='center', fontweight='bold')

axes[1].pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, explode=(0.05, 0, 0))
axes[1].set_title('Outcome Proportions')

plt.suptitle(f'Rock-Paper-Scissors Simulation  |  mode={MODE}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('rock_paper_scissors_simulation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved simulation chart → rock_paper_scissors_simulation.png')

Saved simulation chart → rock_paper_scissors_simulation.png


### Compare strategies side-by-side

In [13]:
strategies = ['always_rock', 'always_paper', 'random', 'cycle']
print('Strategy comparison (2000 games each, classic):')
for strat in strategies:
    c = run_simulation(n_games=2000, strategy=strat, mode='classic', seed=42)
    tot = sum(c.values())
    print(f"  {strat:12s} → win {100*c['win']/tot:4.1f}%  lose {100*c['lose']/tot:4.1f}%  tie {100*c['tie']/tot:4.1f}%")

Strategy comparison (2000 games each, classic):
  always_rock   → win 33.0%  lose 33.9%  tie 33.1%
  always_paper  → win 33.5%  lose 33.0%  tie 33.5%
  random        → win 33.4%  lose 33.2%  tie 33.4%
  cycle         → win 33.2%  lose 33.5%  tie 33.3%


## 10. Exploring the Original Questions (from the book)

1. Changing `random.randint(1, 3)` to `random.randint(1, 300)` leaves most random numbers
   without a matching `if` branch, so `computerMove` stays undefined → NameError when printed.

2. Changing the tie test to `True` forces every round to be recorded as a tie (the later
   win/lose branches become unreachable).

## Key Takeaways

- The classic game is a short loop of input → map → countdown → random opponent → if-elif outcome.
- Replacing the long if-elif with a `BEATS` dictionary makes the logic far easier to extend (RPSLS).
- Dramatic `time.sleep` pauses improve user experience with almost no extra code.
- Simulations reveal that pure random play converges to ~33 % win / lose / tie (classic).
- Adding points + double-or-nothing or best-of-N turns a toy into a richer mini-game.